In [1]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip uninstall -y pyspark -q
!pip install -q pyspark==3.4.1 findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.1 which is incompatible.


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

In [3]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("HospitalAnalytics").master("local[*]").getOrCreate()
print("Spark Started Successfully")
print("Version:", spark.version)

Spark Started Successfully
Version: 3.4.1


In [4]:
patients_csv = """patient_id,name,city,age,gender,registration_date
1,Aarav,Hyderabad,29,Male,2023-01-10
2,Priya,Bangalore,34,Female,2023-02-12
3,Rahul,Mumbai,41,Male,2023-03-14
4,Sneha,Delhi,26,Female,2023-04-15
5,Kiran,Chennai,37,Male,2023-05-11
6,Meera,Hyderabad,31,Female,2023-06-10
7,Amit,Pune,45,Male,2023-06-22
8,Neha,Delhi,28,Female,2023-07-10
9,Divya,Bangalore,33,Female,2023-07-15
10,Vikram,Mumbai,52,Male,2023-08-01
11,Farhan,Hyderabad,39,Male,2023-08-10
12,Simran,Delhi,25,Female,2023-08-21
"""

doctors_csv = """doctor_id,doctor_name,specialization,city,consultation_fee
101,Dr Sharma,Cardiology,Hyderabad,1200
102,Dr Iyer,Dermatology,Bangalore,800
103,Dr Khan,Orthopedics,Mumbai,1500
104,Dr Reddy,Pediatrics,Delhi,900
105,Dr Mehta,Neurology,Hyderabad,2000
106,Dr Nair,Cardiology,Chennai,1300
107,Dr Verma,Dermatology,Pune,850
108,Dr Rao,Orthopedics,Delhi,1400
"""

appointments_csv = """appointment_id,patient_id,doctor_id,appointment_date,status
1,1,101,2024-03-01,Completed
2,2,102,2024-03-01,Completed
3,3,103,2024-03-02,Completed
4,4,104,2024-03-02,Pending
5,5,106,2024-03-03,Completed
6,6,105,2024-03-03,Completed
7,7,107,2024-03-04,Cancelled
8,8,108,2024-03-04,Completed
9,9,102,2024-03-05,Completed
10,10,103,2024-03-05,Completed
11,11,101,2024-03-06,Pending
12,12,104,2024-03-06,Completed
13,1,105,2024-03-07,Completed
14,3,108,2024-03-07,Completed
15,6,101,2024-03-08,Cancelled
16,9,106,2024-03-08,Completed
"""

bills_csv = """bill_id,appointment_id,bill_amount,payment_mode,payment_status
1,1,1200,UPI,Paid
2,2,800,Credit Card,Paid
3,3,1500,Cash,Paid
4,4,900,UPI,Pending
5,5,1300,Debit Card,Paid
6,6,2000,Credit Card,Paid
7,7,850,Cash,Cancelled
8,8,1400,UPI,Paid
9,9,800,UPI,Paid
10,10,1500,Credit Card,Paid
11,11,1200,UPI,Pending
12,12,900,Cash,Paid
13,13,2000,Credit Card,Paid
14,14,1400,UPI,Paid
15,15,1200,Cash,Cancelled
16,16,1300,Debit Card,Paid
"""

hospital_logs_txt = """Aarav login
Priya login
Rahul appointment
Sneha login
Aarav payment
Kiran appointment
Meera login
Vikram appointment
Divya payment
Farhan login
Simran appointment
Neha payment
Amit login
Rahul payment
Meera appointment
Aarav logout
Priya payment
Divya login
Vikram payment
Farhan appointment
"""

patient_profiles_json = """[
  {
    "patient_id": 1,
    "name": "Aarav",
    "contact": {"email": "aarav@mail.com", "phone": "9000011111"},
    "allergies": ["Dust", "Peanuts"]
  },
  {
    "patient_id": 2,
    "name": "Priya",
    "contact": {"email": "priya@mail.com", "phone": "9000022222"},
    "allergies": ["Pollen"]
  },
  {
    "patient_id": 3,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000033333"},
    "allergies": ["Dust", "Milk"]
  },
  {
    "patient_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "allergies": ["Seafood"]
  },
  {
    "patient_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "allergies": ["Pollen", "Dust"]
  }
]"""

with open("patients.csv", "w") as f:
    f.write(patients_csv)

with open("doctors.csv", "w") as f:
    f.write(doctors_csv)

with open("appointments.csv", "w") as f:
    f.write(appointments_csv)

with open("bills.csv", "w") as f:
    f.write(bills_csv)

with open("hospital_logs.txt", "w") as f:
    f.write(hospital_logs_txt)

with open("patient_profiles.json", "w") as f:
    f.write(patient_profiles_json)

print("Hospital datasets created successfully")

Hospital datasets created successfully


In [5]:
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors = spark.read.csv("doctors.csv", header=True, inferSchema=True)
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
bills = spark.read.csv("bills.csv", header=True, inferSchema=True)
profiles = spark.read.json("patient_profiles.json", multiLine=True)

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [7]:
#1
patients.show()

#2
doctors.show()

#3
appointments.show()

#4
bills.show()

#5
patients.printSchema()

#6
doctors.printSchema()

#7
appointments.printSchema()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+

+---------+-----------+--------------+---------+-------

In [8]:
#8
patients.count()

12

In [9]:
#9
doctors.count()

8

In [10]:
#10
appointments.show(5)

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
+--------------+----------+---------+----------------+---------+
only showing top 5 rows



In [11]:
#11
patients.select("name","city","age").show()

#12
doctors.select("doctor_name","specialization","consultation_fee").show()

#13
patients.withColumnRenamed("name","patient_name").show()

#14
doctors.withColumnRenamed("doctor_name","consultant_name").show()

#15
patients.filter(col("city")=="Hyderabad").show()

#16
patients.filter(col("gender")=="Female").show()

#17
patients.filter(col("age")>35).show()

#18
doctors.filter(col("city")=="Hyderabad").show()

#19
doctors.filter(col("specialization")=="Cardiology").show()

#20
doctors.filter(col("consultation_fee")>1000).show()

+------+---------+---+
|  name|     city|age|
+------+---------+---+
| Aarav|Hyderabad| 29|
| Priya|Bangalore| 34|
| Rahul|   Mumbai| 41|
| Sneha|    Delhi| 26|
| Kiran|  Chennai| 37|
| Meera|Hyderabad| 31|
|  Amit|     Pune| 45|
|  Neha|    Delhi| 28|
| Divya|Bangalore| 33|
|Vikram|   Mumbai| 52|
|Farhan|Hyderabad| 39|
|Simran|    Delhi| 25|
+------+---------+---+

+-----------+--------------+----------------+
|doctor_name|specialization|consultation_fee|
+-----------+--------------+----------------+
|  Dr Sharma|    Cardiology|            1200|
|    Dr Iyer|   Dermatology|             800|
|    Dr Khan|   Orthopedics|            1500|
|   Dr Reddy|    Pediatrics|             900|
|   Dr Mehta|     Neurology|            2000|
|    Dr Nair|    Cardiology|            1300|
|   Dr Verma|   Dermatology|             850|
|     Dr Rao|   Orthopedics|            1400|
+-----------+--------------+----------------+

+----------+------------+---------+---+------+-----------------+
|patient_id|p

In [12]:
#21
patients.orderBy("age").show()

#22
patients.orderBy(col("age").desc()).show()

#23
patients.orderBy(col("age").desc()).show(5)

#24
patients.orderBy("age").show(3)

#25
doctors.orderBy(col("consultation_fee").desc()).show()

#26
doctors.orderBy(col("consultation_fee").desc()).show(3)

#27
doctors.orderBy("consultation_fee").show()

#28
appointments.orderBy("appointment_date").show()

#29
bills.orderBy(col("bill_amount").desc()).show()

#30
bills.orderBy(col("bill_amount").desc()).show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+------+---------+---+------+-----------------+

+----------+------+---------+---+------+---------------

In [13]:
#31
patients.groupBy("city").count().show()

#32
patients.groupBy("gender").count().show()

#33
doctors.groupBy("specialization").count().show()

#34
patients.select(avg("age")).show()

#35
patients.select(max("age")).show()

#36
patients.select(min("age")).show()

#37
doctors.select(avg("consultation_fee")).show()

#38
doctors.select(max("consultation_fee")).show()

#39
bills.select(sum("bill_amount")).show()

#40
bills.select(avg("bill_amount")).show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+------+-----+
|gender|count|
+------+-----+
|Female|    6|
|  Male|    6|
+------+-----+

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    1|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    2|
+--------------+-----+

+--------+
|avg(age)|
+--------+
|    35.0|
+--------+

+--------+
|max(age)|
+--------+
|      52|
+--------+

+--------+
|min(age)|
+--------+
|      25|
+--------+

+---------------------+
|avg(consultation_fee)|
+---------------------+
|              1243.75|
+---------------------+

+---------------------+
|max(consultation_fee)|
+---------------------+
|                 2000|
+---------------------+

+----------------+
|sum(bill_amount)|
+----------------+
|           20250|
+----------------+

+-------------

In [14]:
#41
patients.groupBy("city").count().show()

#42
doctors.groupBy("city").count().show()

#43
appointments.groupBy("status").count().show()

#44
bills.groupBy("payment_status").sum("bill_amount").show()

#45
bills.groupBy("payment_mode").sum("bill_amount").show()

#46
bills.groupBy("payment_mode").avg("bill_amount").show()

#47
appointments.groupBy("doctor_id").count().show()

#48
appointments.groupBy("patient_id").count().show()

#49
bills.groupBy("appointment_id").sum("bill_amount").show()

#50
doctors.groupBy("specialization").avg("consultation_fee").show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|  Chennai|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+

+---------+-----+
|   status|count|
+---------+-----+
|Completed|   12|
|Cancelled|    2|
|  Pending|    2|
+---------+-----+

+--------------+----------------+
|payment_status|sum(bill_amount)|
+--------------+----------------+
|     Cancelled|            2050|
|       Pending|            2100|
|          Paid|           16100|
+--------------+----------------+

+------------+----------------+
|payment_mode|sum(bill_amount)|
+------------+----------------+
| Credit Card|            6300|
|        Cash|            4450|
|  Debit Card|            2600|
|         UPI|            6900|
+------------+----------------+

+------------+-

In [15]:
#51
patients.join(appointments, "patient_id").show()

#52
patients.join(appointments, "patient_id").select("name", "city", "appointment_date", "status").show()

#53
doctors.join(appointments, "doctor_id").show()

#54
doctors.join(appointments, "doctor_id").select("doctor_name", "specialization", "appointment_date", "status").show()

#55
appointments.join(bills, "appointment_id").show()

#56
appointments.join(bills, "appointment_id").select("appointment_id", "status", "bill_amount", "payment_status").show()

#57
patients.join(appointments, "patient_id").join(doctors, "doctor_id").show()

#58
patients.join(appointments, "patient_id").join(doctors, "doctor_id").select("name", "doctor_name", "specialization", "appointment_date").show()

#59
patients.join(appointments, "patient_id").join(doctors, "doctor_id").join(bills, "appointment_id").show()

#60
patients.join(appointments, "patient_id").join(doctors, "doctor_id").join(bills, "appointment_id").select("name", "doctor_name", "status", "bill_amount", "payment_mode").show()

+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|doctor_id|appointment_date|   status|
+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|      101|      2024-03-01|Completed|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|      102|      2024-03-01|Completed|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|      103|      2024-03-02|Completed|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             4|      104|      2024-03-02|  Pending|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             5|      106|      2024-03-03|Completed|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|             6|      105|      2024-03-03|Completed|
|

In [16]:
#61
patients.withColumn("age_group",when(col("age") < 30, "Young").when(col("age") < 45, "Adult").otherwise("Senior")).show()

#62
patients.withColumn("hospital_name", lit("BotCampus Hospital")).show()

#63
doctors.withColumn("fee_with_tax",col("consultation_fee") * 1.18).show()

#64
bills.withColumn("bill_with_tax",col("bill_amount") * 1.18).show()

#65
bills.withColumn("bill_in_thousands",col("bill_amount") / 1000).show()

#66
patients.withColumn("country", lit("India")).show()

#67
doctors.withColumn("doctor_label",concat_ws(" - ", "doctor_name", "specialization")).show()

#68
patients.withColumn("patient_label",concat_ws(" - ", "name", "city")).show()

#69
bills.withColumn("high_bill_flag",when(col("bill_amount") > 1500, True).otherwise(False)).show()

#70
patients.withColumn("senior_patient_flag",when(col("age") > 40, True).otherwise(False)).show()

+----------+------+---------+---+------+-----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|age_group|
+----------+------+---------+---+------+-----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    Adult|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   Senior|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    Adult|
|        12|Simran|    Delhi| 25|F

In [17]:
#71
patients.withColumn("patient_age_category", when(col("age") < 30, "Young").when(col("age") < 45, "Adult").otherwise("Senior")).show()

#72
bills.withColumn("bill_category", when(col("bill_amount") >= 1500, "High").when(col("bill_amount") >= 1000, "Medium").otherwise("Low")).show()

#73
doctors.withColumn("doctor_fee_category", when(col("consultation_fee") >= 1500, "Premium").when(col("consultation_fee") >= 1000, "Standard").otherwise("Basic")).show()

#74
appointments.withColumn("appointment_priority", when(col("status") == "Pending", "High").when(col("status") == "Completed", "Normal").otherwise("Low")).show()

#75
patients.withColumn("city_zone", when(col("city").isin("Hyderabad","Bangalore","Chennai"), "South").when(col("city").isin("Mumbai","Delhi","Pune"), "Metro")).show()

+----------+------+---------+---+------+-----------------+--------------------+
|patient_id|  name|     city|age|gender|registration_date|patient_age_category|
+----------+------+---------+---+------+-----------------+--------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|               Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|               Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|               Adult|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|               Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|               Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|               Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|              Senior|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|               Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|               Adult|
|        10|Vikram|   Mumbai| 52|  Male|

In [18]:
#76
patients = patients.withColumn("registration_date", to_date("registration_date"))

#77
patients.withColumn("registration_year", year("registration_date")).show()

#78
patients.withColumn("registration_month", month("registration_date")).show()

#79
appointments = appointments.withColumn("appointment_date", to_date("appointment_date"))

#80
appointments.withColumn("appointment_month", month("appointment_date")).show()

#81
appointments.groupBy("appointment_date").count().show()

#82
appointments.groupBy(month("appointment_date")).count().show()

#83
patients.filter(col("registration_date") > "2023-06-01").show()

#84
patients.withColumn("days_since_registration", datediff(current_date(), col("registration_date"))).show()

#85
appointments.withColumn("days_since_appointment", datediff(current_date(), col("appointment_date"))).show()

+----------+------+---------+---+------+-----------------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|registration_year|
+----------+------+---------+---+------+-----------------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             2023|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2023|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             2023|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             2023|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             2023|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|             2023|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|             2023|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|             2023|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|             2023|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|             2023|

In [19]:
#86
patients.withColumn("rank", rank().over(Window.partitionBy("city").orderBy(col("age").desc()))).show()

#87
patients.withColumn("row_num", row_number().over(Window.partitionBy("city").orderBy(col("age").desc()))).filter(col("row_num")==1).show()

#88
doctors.withColumn("dense_rank", dense_rank().over(Window.partitionBy("specialization").orderBy(col("consultation_fee").desc()))).show()

#89
doctors.withColumn("row_num", row_number().over(Window.partitionBy("specialization").orderBy(col("consultation_fee").desc()))).filter(col("row_num")==1).show()

#90
patients.withColumn("row_num", row_number().over(Window.partitionBy("city").orderBy(col("age").desc()))).filter(col("row_num")<=2).show()

#91
doctors.withColumn("rank", rank().over(Window.orderBy(col("consultation_fee").desc()))).show()

#92
patients.groupBy("city").count().withColumn("rank", rank().over(Window.orderBy(col("count").desc()))).show()

#93
appointments.groupBy("doctor_id").count().withColumn("rank", rank().over(Window.orderBy(col("count").desc()))).show()

#94
bills.withColumn("running_total", sum("bill_amount").over(Window.partitionBy("payment_mode").orderBy("bill_id"))).show()

#95
appointments.withColumn("running_count", count("appointment_id").over(Window.partitionBy("doctor_id").orderBy("appointment_id"))).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|rank|
+----------+------+---------+---+------+-----------------+----+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|   1|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|   2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|   1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|   2|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|   3|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|   1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|   2|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|   3|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   1|
+----------+------+---------+---+------+

In [24]:
logs = spark.sparkContext.textFile("hospital_logs.txt")

#96
logs.count()

20

In [25]:
#97
logs.map(lambda x: x.split()[0]).collect()

['Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Aarav',
 'Kiran',
 'Meera',
 'Vikram',
 'Divya',
 'Farhan',
 'Simran',
 'Neha',
 'Amit',
 'Rahul',
 'Meera',
 'Aarav',
 'Priya',
 'Divya',
 'Vikram',
 'Farhan']

In [26]:
#98
logs.map(lambda x: x.split()[1]).collect()

['login',
 'login',
 'appointment',
 'login',
 'payment',
 'appointment',
 'login',
 'appointment',
 'payment',
 'login',
 'appointment',
 'payment',
 'login',
 'payment',
 'appointment',
 'logout',
 'payment',
 'login',
 'payment',
 'appointment']

In [27]:
#99
logs.map(lambda x: x.split()[0]).distinct().collect()

['Kiran',
 'Vikram',
 'Divya',
 'Simran',
 'Amit',
 'Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Meera',
 'Farhan',
 'Neha']

In [28]:
#100
logs.map(lambda x: (x.split()[1], 1)).reduceByKey(lambda a, b: a + b).collect()

[('login', 7), ('payment', 6), ('appointment', 6), ('logout', 1)]

In [31]:
# BONUS JSON
profiles = spark.read.json("patient_profiles.json", multiLine=True)

#1
profiles.show()

#2
profiles.printSchema()

#3
profiles.select("patient_id","name",col("contact.email").alias("email"),col("contact.phone").alias("phone")).show()

#4
profiles.select("patient_id","name",explode("allergies").alias("allergy")).show()

#5
profiles.select(explode("allergies").alias("allergy")).groupBy("allergy").count().show()

#6
profiles.select("patient_id","name",explode("allergies").alias("allergy")).filter(col("allergy")=="Dust").show()

#7
profiles.select(explode("allergies").alias("allergy")).distinct().show()

#8
profiles.join(patients,"patient_id").show()

#9
profiles.join(patients,"patient_id").select(profiles["name"], patients["city"], col("contact.email").alias("email"), explode("allergies").alias("allergy")).show()

#10
profiles.join(patients,"patient_id").select(patients["city"], explode("allergies").alias("allergy")).groupBy("city").count().show()

+---------------+--------------------+------+----------+
|      allergies|             contact|  name|patient_id|
+---------------+--------------------+------+----------+
|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav|         1|
|       [Pollen]|{priya@mail.com, ...| Priya|         2|
|   [Dust, Milk]|{rahul@mail.com, ...| Rahul|         3|
|      [Seafood]|{meera@mail.com, ...| Meera|         6|
| [Pollen, Dust]|{vikram@mail.com,...|Vikram|        10|
+---------------+--------------------+------+----------+

root
 |-- allergies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- name: string (nullable = true)
 |-- patient_id: long (nullable = true)

+----------+------+---------------+----------+
|patient_id|  name|          email|     phone|
+----------+------+---------------+----------+
|         1| Aarav| aarav@mail.com|9000011111|
|    

In [34]:
# BONUS SQL
patients.createOrReplaceTempView("patients")
doctors.createOrReplaceTempView("doctors")
appointments.createOrReplaceTempView("appointments")
bills.createOrReplaceTempView("bills")

#1
spark.sql("SELECT * FROM patients").show()

#2
spark.sql("SELECT * FROM patients WHERE city='Hyderabad'").show()

#3
spark.sql("SELECT city, COUNT(*) AS total_patients FROM patients GROUP BY city").show()

#4
spark.sql("SELECT specialization, COUNT(*) AS total_doctors FROM doctors GROUP BY specialization").show()

#5
spark.sql("SELECT * FROM patients p JOIN appointments a ON p.patient_id=a.patient_id").show()

#6
spark.sql("SELECT * FROM doctors d JOIN appointments a ON d.doctor_id=a.doctor_id").show()

#7
spark.sql("SELECT * FROM appointments a JOIN bills b ON a.appointment_id=b.appointment_id").show()

#8
spark.sql("SELECT payment_mode, SUM(bill_amount) AS total_bill FROM bills GROUP BY payment_mode").show()

#9
spark.sql("SELECT doctor_id, doctor_name, consultation_fee, RANK() OVER(ORDER BY consultation_fee DESC) AS rank FROM doctors").show()

#10
spark.sql("SELECT p.name, SUM(b.bill_amount) AS total_bill FROM patients p JOIN appointments a ON p.patient_id=a.patient_id JOIN bills b ON a.appointment_id=b.appointment_id GROUP BY p.name ORDER BY total_bill DESC LIMIT 1").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+

+----------+------+---------+---+------+---------------